# Spectra & summary statistics
Beyond scalar moments, the {class}`~phasegen.distributions.Coalescent` provides full spectra, namely the joint (multi-population) and two-locus site-frequency spectra, as well as a set of standard scalar summary statistics. All of these are exact and respect the full demography and coalescent model. See the {doc}`quickstart` for how to configure the underlying model.

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
import phasegen as pg
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
library(phasegen)
pg <- load_phasegen()
```

~~~~~

~~~~~~


## Joint site-frequency spectrum
For multiple populations, {attr}`~phasegen.distributions.Coalescent.jsfs` gives the joint (multi-population) SFS: the expected branch length subtending each configuration of derived-allele counts per population (the deme of origin). For ``P`` populations it is a ``P``-dimensional {class}`~sfsutils.spectrum.JointSFS` of shape ``(n_0 + 1, ..., n_{P-1} + 1)``, with higher moments available via {meth}`moment(k) <phasegen.distributions.JointSFSDistribution.moment>`, {attr}`var <phasegen.distributions.JointSFSDistribution.var>` and {attr}`cov <phasegen.distributions.JointSFSDistribution.cov>`. It is restricted to a single locus, and the state space grows quickly with the per-population sample sizes, which therefore need to remain small.

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
# a two-population demography with a population-size change and asymmetric migration
coal = pg.Coalescent(
    n={'pop_0': 5, 'pop_1': 5},
    demography=pg.Demography(
        pop_sizes={'pop_0': {0: 1, 1: 0.3}, 'pop_1': {0: 1.5}},
        migration_rates={('pop_0', 'pop_1'): 0.5, ('pop_1', 'pop_0'): 0.2},
    ),
)

# mean joint SFS: expected branch length subtending each (pop_0, pop_1) allele-frequency configuration
coal.jsfs.mean.plot_surface(title='Mean joint SFS');
```

```{glue} python-s1-0-0
```

```{glue} python-s1-0-1
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
# a two-population demography with a population-size change and asymmetric migration
coal <- pg$Coalescent(
    n = list(pop_0 = 5, pop_1 = 5),
    demography = pg$Demography(
        pop_sizes = list(pop_0 = 1, pop_1 = 1.5),
        events = c(
            pg$PopSizeChange(pop = "pop_0", time = 1, size = 0.3),
            pg$MigrationRateChange(source = "pop_0", dest = "pop_1", time = 0, rate = 0.5),
            pg$MigrationRateChange(source = "pop_1", dest = "pop_0", time = 0, rate = 0.2)
        )
    )
)

# mean joint SFS: expected branch length subtending each (pop_0, pop_1) allele-frequency configuration
persp(coal$jsfs$mean, title = "Mean joint SFS")
```

```{glue} r-s1-0-0
```

```{glue} r-s1-0-1
```

~~~~~

~~~~~~


## Tree height under recombination
For two loci separated by recombination, ``phasegen`` provides the exact distribution of the tree height, covering both the time to the ultimate MRCA across the two loci and the marginal genealogy at each locus.

Two loci at recombination rate 0.1, and the mean time to the ultimate most recent common ancestor of the sample across both loci:

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
coal = pg.Coalescent(n=8, loci=pg.LocusConfig(recombination_rate=0.1, n=2))

coal.tree_height.mean
```

```{glue} python-s2-0-0
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
coal <- pg$Coalescent(n = 8L, loci = pg$LocusConfig(recombination_rate = 0.1, n = 2L))

coal$tree_height$mean
```

```{glue} r-s2-0-0
```

~~~~~

~~~~~~


The mean marginal tree height of the first locus:

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
coal.tree_height.loci[0].mean
```

```{glue} python-s3-0-0
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
coal$tree_height$loci[[0L]]$mean
```

```{glue} r-s3-0-0
```

~~~~~

~~~~~~


The same mean when all eight lineages start out unlinked between the two loci:

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
coal = pg.Coalescent(n=8, loci=pg.LocusConfig(recombination_rate=0.1, n=2, n_unlinked=8))

coal.tree_height.mean
```

```{glue} python-s4-0-0
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
coal <- pg$Coalescent(n = 8L, loci = pg$LocusConfig(recombination_rate = 0.1, n = 2L, n_unlinked = 8L))

coal$tree_height$mean
```

```{glue} r-s4-0-0
```

~~~~~

~~~~~~


## Two-locus SFS under recombination
For two loci separated by recombination rate ``r``, {attr}`~phasegen.distributions.Coalescent.sfs2` gives the two-locus SFS, whose entry ``(i, j)`` is the expected product of the branch length subtending ``i`` samples at locus 0 and ``j`` samples at locus 1. Its mean, higher moments via {meth}`moment(k) <phasegen.distributions.TwoLocusSFSDistribution.moment>`, {attr}`var <phasegen.distributions.TwoLocusSFSDistribution.var>` and the correlation are {class}`~sfsutils.spectrum.TwoLocusSFS` objects, and {meth}`accumulate() <phasegen.distributions.TwoLocusSFSDistribution.accumulate>` returns an array with one ``(n + 1, n + 1)`` slice per end time. The mean interpolates between the within-tree second moment ``E[L_i L_j]`` at ``r = 0`` (fully linked) and the product of the single-locus means as ``r → ∞`` (for the standard coalescent). The starting linkage is set via the {class}`~phasegen.locus.LocusConfig` ``n_unlinked``. A single population is supported, and the state space grows quickly with the sample size.

The single- and two-locus spectra apply to different locus configurations. {attr}`~phasegen.distributions.Coalescent.sfs2` requires exactly two loci, while the single-locus {attr}`~phasegen.distributions.Coalescent.sfs` requires one. Its marginal mean does not depend on the recombination rate, so the SFS of one of two loci equals that of a coalescent with a single locus.

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
import matplotlib.pyplot as plt

# the two-locus SFS interpolates between tightly linked and independent loci as r grows
_, axs = plt.subplots(ncols=2, figsize=(7, 3.4), subplot_kw={"projection": "3d"})

# r = 0.1: tightly linked -> strong cross-locus structure (close to the within-tree SFS covariance)
pg.Coalescent(n=6, loci=2, recombination_rate=0.1).sfs2.mean.plot_surface(
    ax=axs[0], show=False, title='Tightly linked (r = 0.1)')

# r = 10: nearly independent -> approaches the outer product of the marginal SFS
pg.Coalescent(n=6, loci=2, recombination_rate=10.0).sfs2.mean.plot_surface(
    ax=axs[1], title='Near-independent (r = 10)');
```

```{glue} python-s5-0-0
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
# the two-locus SFS interpolates between tightly linked and independent loci as r grows

par(mfrow = c(1, 2))
# r = 0.1: tightly linked -> strong cross-locus structure (close to the within-tree SFS covariance)
persp(pg$Coalescent(n = 6L, loci = 2L, recombination_rate = 0.1)$sfs2$mean, title = "Tightly linked (r = 0.1)")

# r = 10: nearly independent -> approaches the outer product of the marginal SFS
persp(pg$Coalescent(n = 6L, loci = 2L, recombination_rate = 10.0)$sfs2$mean, title = "Near-independent (r = 10)")
```

```{glue} r-s5-0-0
```

~~~~~

~~~~~~


## Summary statistics
Beyond full spectra, several standard scalar summaries are available directly from the {class}`~phasegen.distributions.Coalescent`, each respecting the full demography and coalescent model:

- Population structure: Hudson's {attr}`Coalescent.fst <phasegen.distributions.Coalescent.fst>` and Patterson's f-statistics ({meth}`~phasegen.distributions.Coalescent.f2`, {meth}`~phasegen.distributions.Coalescent.f3`, {meth}`~phasegen.distributions.Coalescent.f4`), all derived from inter-population pairwise coalescence times.
- Linkage: the correlation of coalescence times between two loci ({meth}`tree_height.loci.get_corr <phasegen.distributions.MarginalLocusDistributions.get_corr>`), which decays towards zero as the recombination rate grows (for the standard coalescent).
- SFS skew: Tajima's {attr}`~phasegen.distributions.UnfoldedSFSDistribution.tajimas_d`, together with the underlying {attr}`~phasegen.distributions.UnfoldedSFSDistribution.theta_pi` and {attr}`~phasegen.distributions.UnfoldedSFSDistribution.theta_w` estimators.

We illustrate them on a relatively complex scenario: a structured three-population demography with asymmetric population sizes and migration.

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
struct = pg.Coalescent(
    n={'pop_0': 2, 'pop_1': 2, 'pop_2': 2},
    demography=pg.Demography(
        pop_sizes={'pop_0': 1.0, 'pop_1': 1.0, 'pop_2': 1.5},
        migration_rates={
            ('pop_0', 'pop_1'): 0.5, ('pop_1', 'pop_0'): 0.5,
            ('pop_1', 'pop_2'): 0.2, ('pop_2', 'pop_1'): 0.2,
            ('pop_0', 'pop_2'): 0.2, ('pop_2', 'pop_0'): 0.2,
        },
    ),
)
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
struct <- pg$Coalescent(
    n = list(pop_0 = 2, pop_1 = 2, pop_2 = 2),
    demography = pg$Demography(
        pop_sizes = list(pop_0 = 1.0, pop_1 = 1.0, pop_2 = 1.5),
        events = c(
            pg$SymmetricMigrationRateChanges(pops = c("pop_0", "pop_1"), rate = 0.5),
            pg$SymmetricMigrationRateChanges(pops = c("pop_1", "pop_2"), rate = 0.2),
            pg$SymmetricMigrationRateChanges(pops = c("pop_0", "pop_2"), rate = 0.2)
        )
    )
)
```

~~~~~

~~~~~~


Population structure, as Hudson's F<sub>ST</sub> and Patterson's f-statistics:

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
print(f"F_ST                    = {struct.fst:.3f}")
print(f"f2(pop_0, pop_2)        = {struct.f2('pop_0', 'pop_2'):.3f}")
print(f"f3(pop_1; pop_0, pop_2) = {struct.f3('pop_1', 'pop_0', 'pop_2'):.3f}")
```

```{glue} python-s7-0-0
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
cat(sprintf("F_ST                    = %.3f\n", struct$fst))
cat(sprintf("f2(pop_0, pop_2)        = %.3f\n", struct$f2("pop_0", "pop_2")))
cat(sprintf("f3(pop_1; pop_0, pop_2) = %.3f\n", struct$f3("pop_1", "pop_0", "pop_2")))
```

```{glue} r-s7-0-0
```

~~~~~

~~~~~~


Linkage, as the correlation of the coalescence times at two loci, which decays as the recombination rate grows:

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
for r in [0.1, 1.0, 10.0]:
    corr = pg.Coalescent(n=2, loci=2, recombination_rate=r).tree_height.loci.get_corr(0, 1)
    print(f"corr(T_A, T_B) at r={r:<4} = {corr:.3f}")
```

```{glue} python-s8-0-0
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
for (r in c(0.1, 1.0, 10.0)) {
    corr <- pg$Coalescent(n = 2L, loci = 2L, recombination_rate = r)$tree_height$loci$get_corr(0L, 1L)
    cat(sprintf("corr(T_A, T_B) at r=%-4s = %.3f\n", r, corr))
}
```

```{glue} r-s8-0-0
```

~~~~~

~~~~~~


Skew of the SFS, as Tajima's D under recent population growth, where an excess of low-frequency variants makes D negative:

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
growth = pg.Coalescent(n=10, demography=pg.Demography(pop_sizes={'pop_0': {0: 1.0, 0.5: 0.1}}))
print(f"Tajima's D (growth) = {growth.sfs.tajimas_d:.3f}")
```

```{glue} python-s9-0-0
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
growth <- pg$Coalescent(
    n = 10L,
    demography = pg$Demography(
        pop_sizes = list(pop_0 = 1.0),
        events = c(pg$PopSizeChange(pop = "pop_0", time = 0.5, size = 0.1))
    )
)
cat(sprintf("Tajima's D (growth) = %.3f\n", growth$sfs$tajimas_d))
```

```{glue} r-s9-0-0
```

~~~~~

~~~~~~
